# Phase 7: Ligand–Receptor Communication & TME Received Signal Analysis

### Overview
In this analysis, we characterize cell–cell communication between the tumour microenvironment (TME) lineages (T-cells, B-cells, Myeloid, NK cells, Plasma cells, Fibroblasts, Endothelial) and **Thyrocytes** across thyroid cancer progression stages.

We calculate:
1. Canonical ligand–receptor interaction scores (Mean Ligand in Sender x Mean Receptor in Target) across all cell lineages.
2. Pathway-level aggregation for key oncogenic & microenvironmental cascades (TGF-beta, Chemokine, Integrin/ECM, ErbB/EGFR, Immune/Stromal checkpoint, etc.).
3. Thyrocyte-specific **'Received Signal'** scores per cell, mapping microenvironmental pressure onto thyrocyte differentiation states.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata

PROCESSED_DIR = os.path.join(os.getcwd(), "processed_data")

comm_df = pd.read_csv(os.path.join(PROCESSED_DIR, "ligand_receptor_communication_atlas.csv"))
thy_signals = pd.read_csv(os.path.join(PROCESSED_DIR, "thyrocyte_received_signals_by_pathway.csv"))
print(f"Loaded {len(comm_df)} ligand-receptor interactions across atlas lineages.")
comm_df.head(10)


### Cell 2: TME-to-Thyrocyte Communication Landscape
We filter for signals targeted specifically to **Thyrocytes** to assess which microenvironmental cells exert the strongest signaling pressure on malignant and premalignant thyroid follicular cells.


In [ ]:
top_thy_signals = comm_df[comm_df["target"] == "Thyrocytes"].sort_values("communication_score", ascending=False)
print("Top 15 TME -> Thyrocyte Ligand-Receptor Interactions:")
display_cols = ["sender", "ligand", "receptor", "pathway", "communication_score"]
print(top_thy_signals[display_cols].head(15).to_string(index=False))


### Cell 3: Pathway Received Signal Heatmap
We summarize the incoming signal intensity across signaling pathways and sender cell types.


In [ ]:
pivot = thy_signals.pivot(index="pathway", columns="sender", values="communication_score").fillna(0)

plt.figure(figsize=(10, 6.5))
plt.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
plt.xticks(np.arange(len(pivot.columns)), pivot.columns, rotation=45, ha="right", fontsize=11)
plt.yticks(np.arange(len(pivot.index)), pivot.index, fontsize=11)
plt.title("TME-to-Thyrocyte Pathway Received Signal Intensity", fontsize=13, fontweight="bold")
plt.xlabel("Sender Microenvironment Lineage", fontsize=12)
plt.ylabel("Signaling Pathway", fontsize=12)
cbar = plt.colorbar()
cbar.set_label("Communication Score (Mean Ligand x Mean Receptor)", fontsize=11)
plt.tight_layout()
plt.show()


### Cell 4: Single-Cell Received Signal Scores in Thyrocytes
Load per-cell received signal scores and inspect their correlation with differentiation state.


In [ ]:
per_cell_signals = pd.read_csv(os.path.join(PROCESSED_DIR, "thyrocyte_per_cell_received_signals.csv"), index_col=0)
print(f"Per-cell received signal scores computed for {len(per_cell_signals)} thyrocytes across {per_cell_signals.shape[1]} pathways.")
print(per_cell_signals.describe().round(4))
